# Indian Mutual Fund Analytics
Fetches live NAV data from [mfapi.in](https://mfapi.in) and computes 3-year / 5-year CAGR, annualised volatility, and Sharpe ratio.

In [ ]:
# Install dependencies if needed
# !pip install requests pandas numpy

In [ ]:
import requests
import pandas as pd
import numpy as np
from datetime import date, timedelta

RISK_FREE_RATE = 0.065  # 6.5% annual — adjust as needed

## Helper Functions

In [ ]:
def get_last_business_day_of_prev_month():
    today = date.today()
    last = today.replace(day=1) - timedelta(days=1)
    while last.weekday() >= 5:  # skip Saturday/Sunday
        last -= timedelta(days=1)
    return last


def search_fund(query):
    resp = requests.get(f"https://api.mfapi.in/mf/search?q={query}", timeout=10)
    resp.raise_for_status()
    return resp.json()


def get_nav_data(scheme_code):
    resp = requests.get(f"https://api.mfapi.in/mf/{scheme_code}", timeout=10)
    resp.raise_for_status()
    payload = resp.json()
    df = pd.DataFrame(payload["data"])
    df["date"] = pd.to_datetime(df["date"], format="%d-%m-%Y")
    df["nav"] = pd.to_numeric(df["nav"], errors="coerce")
    df = df.dropna(subset=["nav"]).sort_values("date").reset_index(drop=True)
    return df, payload["meta"]


def nearest_nav(df, target):
    subset = df[df["date"] <= pd.Timestamp(target)]
    return float(subset.iloc[-1]["nav"]) if not subset.empty else None


def cagr(nav_start, nav_end, years):
    if nav_start is None or nav_end is None:
        return None
    return (nav_end / nav_start) ** (1 / years) - 1


def annualised_volatility(df, start, end):
    mask = (df["date"] >= pd.Timestamp(start)) & (df["date"] <= pd.Timestamp(end))
    window = df[mask]["nav"].pct_change().dropna()
    return float(window.std() * np.sqrt(252)) if len(window) >= 2 else None


def sharpe(ret, vol, rfr=RISK_FREE_RATE):
    if ret is None or vol is None or vol == 0:
        return None
    return (ret - rfr) / vol

## Input — Enter Fund Name or Scheme Code

In [ ]:
QUERY = "Mirae Asset Large Cap Fund"   # ← change this to a fund name or scheme code

# Resolve to scheme code
try:
    scheme_code = int(QUERY)
    print(f"Using scheme code: {scheme_code}")
except ValueError:
    results = search_fund(QUERY)
    if not results:
        raise ValueError(f"No funds found for '{QUERY}'")
    scheme_code = results[0]["schemeCode"]
    print(f"Matched : {results[0]['schemeName']}")
    print(f"Code    : {scheme_code}")
    if len(results) > 1:
        print(f"(+ {len(results)-1} other match(es) — using top result)")

## Fetch NAV History

In [ ]:
df, meta = get_nav_data(scheme_code)

print(f"Fund      : {meta.get('scheme_name', 'N/A')}")
print(f"Category  : {meta.get('scheme_category', 'N/A')}")
print(f"Fund House: {meta.get('fund_house', 'N/A')}")
print(f"NAV records available: {len(df)}  ({df['date'].min().date()} to {df['date'].max().date()})")
df.tail()

## Calculate Performance Metrics

In [ ]:
base = get_last_business_day_of_prev_month()
nav_base = nearest_nav(df, base)
print(f"Base date : {base.strftime('%d %b %Y')}  |  Base NAV : {nav_base:.4f}")

rows = []
for years in (3, 5):
    try:
        start = base.replace(year=base.year - years)
    except ValueError:
        start = base - timedelta(days=365 * years)

    nav_start = nearest_nav(df, start)
    r = cagr(nav_start, nav_base, years)
    v = annualised_volatility(df, start, base)
    s = sharpe(r, v)

    rows.append({
        "Period": f"{years}-Year",
        "Start Date": start.strftime("%d %b %Y"),
        "Start NAV": round(nav_start, 4) if nav_start else "N/A",
        "CAGR": f"{r*100:.2f}%" if r is not None else "N/A",
        "Volatility (Ann.)": f"{v*100:.2f}%" if v is not None else "N/A",
        "Sharpe Ratio": f"{s:.4f}" if s is not None else "N/A",
    })

summary = pd.DataFrame(rows).set_index("Period")
summary

## NAV Chart

In [ ]:
import matplotlib.pyplot as plt

cutoff = pd.Timestamp(base.replace(year=base.year - 5))
plot_df = df[df["date"] >= cutoff]

plt.figure(figsize=(12, 4))
plt.plot(plot_df["date"], plot_df["nav"], linewidth=1.5)
plt.title(f"{meta.get('scheme_name', 'Fund')} — 5-Year NAV History")
plt.xlabel("Date")
plt.ylabel("NAV (₹)")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()